# Calling our model server

Team 6, floor-plan similarity search · AI Methods I hackathon, IAAC 2026

This notebook calls the **online** model server on Render, like any other program would. It sends one floor plan and prints the five most similar plans the server finds out of the 5,000 of CubiCasa5K.

| Section | What it does |
|---|---|
| 1 | wakes the server up (the free plan sleeps after 15 minutes) |
| 2 | calls the API with `gradio_client`, the easy way |
| 3 | calls it with plain HTTP in two steps, like the Cloudflare app will |
| 4 | shows the five plans that came back |

It only needs `gradio_client`, `requests` and `pillow`: `pip install gradio_client requests pillow`.

In [ ]:
import base64
import io
import json
import time
from pathlib import Path

import requests
from gradio_client import Client
from IPython.display import display
from PIL import Image

MODEL_URL = "https://hackathon-team6-cubicasa.onrender.com"     # our server on Render
EXAMPLE = Path("examples/plan_high_quality_architectural_1191.png")

image_b64 = base64.b64encode(EXAMPLE.read_bytes()).decode()     # HTTP carries text, so the image travels as base64
print(f"{EXAMPLE.name}: {EXAMPLE.stat().st_size / 1024:.1f} KB as a file, {len(image_b64) / 1024:.1f} KB as base64")

## 1. Wake the server up

On the free plan the server falls asleep after 15 minutes without requests. The first request wakes it, which takes about a minute, so we knock until it answers.

In [ ]:
def wake_up(url, tries=10, wait=10):
    for attempt in range(1, tries + 1):
        start = time.time()
        try:
            requests.get(url, timeout=60).raise_for_status()
            print(f"awake: answered in {time.time() - start:.1f} s (attempt {attempt})")
            return True
        except requests.RequestException as error:
            print(f"attempt {attempt}: not yet ({type(error).__name__}), waiting {wait} s")
            time.sleep(wait)
    return False


wake_up(MODEL_URL)

## 2. The easy way: `gradio_client`

`gradio_client` hides the two HTTP steps: we give it the base64 text and get the dictionary back.

In [ ]:
client = Client(MODEL_URL, verbose=False)
start = time.time()
result = client.predict(image_b64, api_name="/predict")
print(f"answered in {time.time() - start:.2f} s")
print(f"reconstruction error: {result['reconstruction_error']}, place on the atlas: {result['atlas_xy']}")
for match in result["matches"]:
    print(f"{match['rank']}. {match['id']:35s} distance {match['distance']:6.2f}")

The example comes from the collection, so the first match is the plan itself, at a much smaller distance than the others. The next four are other homes, at distances around 80 or more: those are the real answers.

## 3. Plain HTTP, in two steps

This is what the Cloudflare app will do on Wednesday, in JavaScript:
1. `POST /gradio_api/call/predict` with the input: the server answers with an `event_id`, a kind of ticket number;
2. `GET /gradio_api/call/predict/<event_id>`: the result comes on the line after `event: complete`.

In [ ]:
ticket = requests.post(f"{MODEL_URL}/gradio_api/call/predict", json={"data": [image_b64]}, timeout=120)
event_id = ticket.json()["event_id"]
print("event_id:", event_id)

answer = requests.get(f"{MODEL_URL}/gradio_api/call/predict/{event_id}", timeout=120).text
print(answer[:160], "...")

result_http = json.loads(answer.split("event: complete\ndata: ", 1)[1].splitlines()[0])[0]
print("same matches as gradio_client:", [m["id"] for m in result_http["matches"]] == [m["id"] for m in result["matches"]])

## 4. The plans that came back

Every match carries its plan as a small PNG (base64), so a page can show it without asking anyone else. Our plan is on the left, then the five matches.

In [ ]:
query = Image.open(EXAMPLE).convert("L").resize((128, 128))
plans = [query] + [Image.open(io.BytesIO(base64.b64decode(m["image_png_base64"]))) for m in result["matches"]]

row = Image.new("L", (len(plans) * 136, 128), 255)
for i, plan in enumerate(plans):
    row.paste(plan, (i * 136, 0))
display(row.resize((row.width * 2, row.height * 2)))
print("left: our plan · then matches 1 to 5:", ", ".join(f"{m['distance']:.1f}" for m in result["matches"]))

## What we see

- The server answers from another program in two ways, with the same five plans.
- Match 1 is the example itself, at a much smaller distance than the others, which shows that the server prepares and cleans the image like the training notebook: a different preprocessing would give a different code.
- The other matches share the footprint and the proportions of the query, whatever the drawing style: since version 2 the server compares the cleaned lines, not how the plan is drawn.
- Every match also has its place on the atlas (`atlas_xy`), and so does our plan: that's what the app will use to show it on the map.